In [5]:
import scipy.io
import numpy as np
from pydub import AudioSegment

In [6]:
# step 1 :load data.
# file of one subject
# data loaded .
file_path = r'E:\HRTF\HRTF_code\cipic-hrtf-database-master\standard_hrir_database\subject_003\hrir_final.mat'
data = scipy.io.loadmat(file_path)

print(data.keys())
left_ear = data['hrir_l'] 
right_ear = data['hrir_r']


# print(data['azimuth'])
# print(data['elevation'])
print(left_ear.shape)
print(right_ear.shape)


dict_keys(['__header__', '__version__', '__globals__', 'OnR', 'OnL', 'ITD', 'hrir_r', 'hrir_l', 'name'])
(25, 50, 200)
(25, 50, 200)


In [7]:
# testing for one direction
filter_l=left_ear[20,49,:]
filter_r=right_ear[20,49,:]

In [8]:
# source audio cleaner funtion

def audio_preprocess(input_file):
    # Read the audio file
    audio = AudioSegment.from_file(input_file)
    sr = audio.frame_rate

    # Resample to 44.1 kHz if needed
    if sr != 44100:
        audio = audio.set_frame_rate(44100)
        sr = 44100

    # Convert to mono
    # Normalize
    audio = audio.set_channels(1)  
    audio = audio.normalize()
    audio.export("sample_audio.wav", format="wav")
    # Convert to numpy
    samples = np.array(audio.get_array_of_samples())

    return samples, sr


# getting the source audio
input_file = r'E:\HRTF\HRTF_code\test_sound.mp3'
clean_audio, sr = audio_preprocess(input_file)

print(clean_audio.shape)
print(sr)
print(clean_audio)

(307993,)
44100
[0 0 0 ... 0 0 0]


In [9]:
#main processing part : convolution of the source audio with the filters for left and right ear

left_ear_audio = np.convolve(clean_audio, filter_l)
right_ear_audio = np.convolve(clean_audio, filter_r)

# testign purposes export the audio stereo audio
stereo_audio = np.stack((left_ear_audio, right_ear_audio), axis=1)
stereo_audio = stereo_audio.astype(np.int16)

# normalize the stereo audio
stereo_audio = stereo_audio / np.max(np.abs(stereo_audio))

# save the stereo audio
stereo_audio_int16 = (stereo_audio * 32767).astype(np.int16)
stereo_audio_segment = AudioSegment(
    stereo_audio_int16.tobytes(),
    frame_rate=sr,
    sample_width=stereo_audio_int16.dtype.itemsize,
    channels=2
)
stereo_audio_segment.export("stereo_output_1.wav", format="wav")


<_io.BufferedRandom name='stereo_output_1.wav'>